Imports

In [1]:
import polars as pl
from zipfile import ZipFile
import numpy as np
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras import layers, models, Input
from sklearn.ensemble import IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
event_path = r"Data\instance_events-000000000000.json.gz"
usage_path = r"Data\instance_usage-000000000000.json.gz"

In [3]:
# Quick debug to see the actual 2019 Usage schema
usage_sample = pl.read_ndjson(usage_path, n_rows=5)
print(usage_sample.columns)
print(usage_sample.head(1))
event_sample = pl.read_ndjson(event_path, n_rows=5)
print(event_sample.columns)
print(event_sample.head(1))

['start_time', 'end_time', 'collection_id', 'instance_index', 'machine_id', 'alloc_collection_id', 'alloc_instance_index', 'collection_type', 'average_usage', 'maximum_usage', 'random_sample_usage', 'assigned_memory', 'page_cache_memory', 'cycles_per_instruction', 'memory_accesses_per_instruction', 'sample_rate', 'cpu_usage_distribution', 'tail_cpu_usage_distribution']
shape: (1, 18)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ start_tim ┆ end_time  ┆ collectio ┆ instance_ ┆ … ┆ memory_ac ┆ sample_ra ┆ cpu_usage ┆ tail_cpu │
│ e         ┆ ---       ┆ n_id      ┆ index     ┆   ┆ cesses_pe ┆ te        ┆ _distribu ┆ _usage_d │
│ ---       ┆ str       ┆ ---       ┆ ---       ┆   ┆ r_instruc ┆ ---       ┆ tion      ┆ istribut │
│ str       ┆           ┆ str       ┆ str       ┆   ┆ tio…      ┆ f64       ┆ ---       ┆ ion      │
│           ┆           ┆           ┆           ┆   ┆ ---       ┆           ┆ list[f64] ┆ ---      │
│      

In [4]:
def extract_metric(col_name, key):
    extracted = (
        pl.col(col_name)
        .cast(pl.Utf8)
        .str.extract(rf"'{key}':\s*([\d\.eE-]+)", 1)
        .cast(pl.Float64, strict=False)
    )
    
    return pl.coalesce([extracted, pl.col(col_name).cast(pl.Float64, strict=False)]).fill_null(0.0)

In [5]:
def load_master_class_2019(event_path, usage_path, n_rows=5000000):
    event_schema = {
        "machine_id": pl.String,
        "time": pl.String,
        "type": pl.String,
        "priority": pl.String,
        "resource_request": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)])
    }
    
    usage_schema = {
        "machine_id": pl.String,
        "start_time": pl.String,
        "average_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
        "maximum_usage": pl.Struct([pl.Field("cpus", pl.Float64), pl.Field("memory", pl.Float64)]),
        "assigned_memory": pl.Float64,
        "cycles_per_instruction": pl.Float64
    }

    events = (
        pl.scan_ndjson(event_path, n_rows=n_rows, schema=event_schema)
        .select([
            pl.col("machine_id").cast(pl.Float64),
            pl.col("time").cast(pl.Float64),
            pl.col("type").cast(pl.Int32).is_in([2, 3, 4]).cast(pl.Int8).alias("error_signal"),
            pl.col("priority").cast(pl.Int32).alias("priority_level"),
            pl.col("resource_request").struct.field("cpus").alias("demand_cpu")
        ])
        .sort("time")
    )

    usage = (
        pl.scan_ndjson(usage_path, n_rows=n_rows, schema=usage_schema)
        .select([
            pl.col("machine_id").cast(pl.Float64),
            pl.col("start_time").cast(pl.Float64).alias("time"),
            pl.col("average_usage").struct.field("cpus").alias("utilization_cpu"),
            pl.col("maximum_usage").struct.field("cpus").alias("saturation_cpu"),
            pl.col("assigned_memory").alias("utilization_mem"),
            pl.col("cycles_per_instruction").alias("delay_cpi")
        ])
        .sort("time")
    )

    return events.join_asof(
        usage, 
        on="time", 
        by="machine_id", 
        strategy="backward"
    ).collect()

df_final = load_master_class_2019(event_path, usage_path)
print(f"Joined Dataset Shape: {df_final.shape}")
df_final

C:\Users\sabbu\AppData\Local\Temp\ipykernel_26736\2215271374.py:49: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  ).collect()


Joined Dataset Shape: (5000000, 9)


machine_id,time,error_signal,priority_level,demand_cpu,utilization_cpu,saturation_cpu,utilization_mem,delay_cpi
f64,f64,i8,i32,f64,f64,f64,f64,f64
null,0.0,0,500,0.096558,null,null,null,null
null,0.0,0,500,0.087769,null,null,null,null
null,0.0,1,500,0.096558,null,null,null,null
null,0.0,1,500,0.087769,null,null,null,null
5.3772e10,0.0,1,500,0.096558,null,null,null,null
…,…,…,…,…,…,…,…,…
null,9.2234e18,0,0,null,null,null,null,null
null,9.2234e18,0,0,null,null,null,null,null
null,9.2234e18,0,0,null,null,null,null,null


In [6]:
def finalize_ml_data(df, window_size=5):
    feature_cols = [
        "demand_cpu", "utilization_cpu", "saturation_cpu", 
        "utilization_mem", "delay_cpi", "priority_level"
    ]
    
    scaler = StandardScaler()
    df_clean = df.fill_null(0.0)
    scaled_data = scaler.fit_transform(df_clean.select(feature_cols).to_numpy())
    
    df_scaled = df_clean.with_columns([
        pl.Series(name=feature_cols[i], values=scaled_data[:, i]) 
        for i in range(len(feature_cols))
    ])

    X, y = [], []
    for _, group in df_scaled.group_by("machine_id"):
        if len(group) <= window_size:
            continue
        
        feat_arr = group.select(feature_cols).to_numpy().astype('float32')
        label_arr = group.select("error_signal").to_numpy().astype('float32')

        for i in range(len(group) - window_size):
            X.append(feat_arr[i : i + window_size])
            y.append(label_arr[i + window_size])
            
    return np.array(X), np.array(y)

X, y = finalize_ml_data(df_final)

print(f"Feature Tensor (X) Shape: {X.shape}")
print(f"Label Array (y) Shape: {y.shape}")

Feature Tensor (X) Shape: (4950077, 5, 6)
Label Array (y) Shape: (4950077, 1)


Test bench Models

In [7]:
def build_CNN_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.Conv1D(64, kernel_size=2, activation='relu'),
        layers.Dropout(0.2),
        layers.Flatten(),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_LSTM_model(input_shape):
    model = models.Sequential([
        layers.Input(shape=input_shape),
        layers.LSTM(64, return_sequences=False),
        layers.Dropout(0.2),
        layers.Dense(32, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])

    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

def build_Hybrid_model(input_shape):
    inputs = Input(shape=input_shape)
    # adding 1d CNN
    x = layers.Conv1D(64, 2, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    #adding LSTM
    x = layers.LSTM(64, return_sequences=True)(x)
    #adding attention
    query_val_attention = layers.Attention()([x,x])
    
    x = layers.GlobalAveragePooling1D()(query_val_attention)
    x = layers.Dense(32, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['AUC'])
    return model

In [ ]:
models_to_test = {
    "Isolation Forest": "baseline_unsupervised",
    "Logistic Reg": "baseline_supervised",
    "Individual CNN": build_CNN_model,
    "Individual LSTM": build_LSTM_model,
    "Elite Hybrid": build_Hybrid_model
}
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train_flat = X_train.reshape(X_train.shape[0], -1)
X_test_flat = X_test.reshape(X_test.shape[0], -1)

early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

results = {}

for name, factory in models_to_test.items():
    print(f"\n--- Training {name} ---")
    if name == "Isolation Forest":
        iso = IsolationForest(contamination=0.05, max_samples=100000, random_state=42).fit(X_train_flat)
        y_pred = [1 if val == -1 else 0 for val in iso.predict(X_test_flat)]
    
    elif name == "Logistic Reg":
        lr = LogisticRegression(max_iter=1000, solver='saga').fit(X_train_flat, y_train.ravel())
        y_pred = lr.predict(X_test_flat)
        
    else:
        m = factory(X_train.shape[1:])
        m.fit(
            X_train, y_train, 
            epochs=10, 
            batch_size=512, 
            verbose=1, 
            validation_split=0.1,
            callbacks=[early_stop]
        )
        y_pred = (m.predict(X_test) > 0.5).astype(int)
    
    score = f1_score(y_test, y_pred)
    results[name] = score
    print(f"{name} F1 Score on Test Set: {score:.4f}")

print("\n" + "="*40)
print(f"PHASE 2 WINNER: {max(results, key=results.get)}")
print("="*40)





--- Training Isolation Forest ---
Isolation Forest F1 Score on Test Set: 0.1158

--- Training Logistic Reg ---
Logistic Reg F1 Score on Test Set: 0.3823

--- Training Individual CNN ---
Epoch 1/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 63s 3ms/step - AUC: 0.7770 - loss: 0.5592 - val_AUC: 0.8116 - val_loss: 0.5231
Epoch 2/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - AUC: 0.8113 - loss: 0.5226 - val_AUC: 0.8189 - val_loss: 0.5141
Epoch 3/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - AUC: 0.8199 - loss: 0.5121 - val_AUC: 0.8284 - val_loss: 0.5016
Epoch 4/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 23s 3ms/step - AUC: 0.8240 - loss: 0.5069 - val_AUC: 0.8330 - val_loss: 0.4968
Epoch 5/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - AUC: 0.8266 - loss: 0.5035 - val_AUC: 0.8323 - val_loss: 0.4973
Epoch 6/10
6962/6962 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - AUC: 0.8288 - loss: 0.5008 - val_AUC: 0.8316 - val_loss: 0.4976
30938/30938 ━━━━━━━━━━━━━━━━━━━━ 19s 626us/step
Individual CNN F1 Score on Test Set: 0.